In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import time
def timeit(f):
    def timed(*args, **kw):
        ts = time.time()
        result = f(*args, **kw)
        te = time.time()
        print('func:%r args:[%r, %r] took: %2.4f sec' % \
          (f.__name__, args, kw, te-ts))
        return result
    return timed

In [ ]:
@timeit
def box_muller_improved(seed, n_samples):
  """
  Runtime of this algorithm is worse than standard BM method
  Acceptance rate: (1 - \pi /4)
  """
  np.random.seed(seed)
  my_samples = np.zeros((n_samples, 2))
  for i in range(n_samples):
    u1u2 = np.ones(2)
    while np.sum(np.square(u1u2)) > 1:
      u1u2 = -1 + 2 * np.random.uniform(size=2)
    S = np.sum(np.square(u1u2),)
    f = np.sqrt(-2*np.log(S)/S)
    my_samples[i] = (u1u2[0]*f, u1u2[1]*f)
  return my_samples
@timeit
def box_muller(seed, n_samples):
  """
  Requires trigonometric functions
  Faster thanks to numpy vectorization
  """
  n_samples =10
  np.random.seed(0)
  u1s = np.random.uniform(size=n_samples)
  u2s = np.random.uniform(size=n_samples)
  f = np.sqrt(-2*np.log(u1s))
  my_samples0 = np.cos(u2s) * f
  my_samples1 = np.sin(u2s) * f
  return my_samples0, my_samples1

In [ ]:
box_muller_improved(0, 1000)
box_muller(0, 1000)

func:'box_muller_improved' args:[(0, 1000), {}] took: 0.1146 sec
func:'box_muller' args:[(0, 1000), {}] took: 0.0001 sec


(array([0.76967824, 0.70691415, 0.84819041, 0.66268661, 1.30729476,
        0.93146088, 1.28541157, 0.32208981, 0.19377803, 0.89281844]),
 array([0.7794797 , 0.41314432, 0.54132631, 0.88046247, 0.09302159,
        0.08136353, 0.0259925 , 0.35404163, 0.19099171, 1.05830598]))

In [ ]:
import jax
import jax.numpy as jnp
from functools import partial

@timeit
@partial(jax.vmap, in_axes=(0, None)) #vmapping over OP_key only
def geometric_sampling(OP_key, p):
  """
  Jax implementation of trial geometric sampling
  """
  def loop_condition(inps):
    _, _, succ = inps
    return ~succ
  def iter(inps):
    n, key, _ = inps
    u = jax.random.uniform(key)
    succ = u < p
    _, key = jax.random.split(key)
    return n+1, key, succ
  n, _, _ = jax.lax.while_loop(loop_condition, iter, (0, OP_key, False))
  return n

OP_key = jax.random.PRNGKey(0)
keys = jax.random.split(OP_key, 100000000) # 100000000 repetitions
ns = geometric_sampling(keys, 0.5)

func:'geometric_sampling' args:[(Array([[2026817864, 3570935637],
       [1078653322,  484567104],
       [1717541691, 1228443415],
       ...,
       [ 908508561, 4040292636],
       [3844487376,  976739947],
       [ 667685819, 3133351912]], dtype=uint32), 0.5), {}] took: 0.8281 sec


In [ ]:
import matplotlib.pyplot as plt
keys = jax.random.split(OP_key, 1000)
ns = geometric_sampling(keys, 0.5)
plt.hist(ns, bins=50)